# Landslide Susceptibility Mapping using Logistic Regression and Gaussian Process
## Cuenca La Iguana, Medellín, Colombia

This notebook implements a two-stage probabilistic framework for landslide susceptibility assessment:

1. **Logistic Regression (LR)**: Baseline discriminative model that uses topographic covariates (slope, aspect, TWI) to estimate $P(\text{landslide}|\mathbf{x})$ at each cell.
2. **Gaussian Process Regression (GP)**: A non-parametric probabilistic model conditioned on the LR-predicted probabilities at observed landslide locations, which propagates uncertainty and captures non-linear covariate structure.

### Workflow
1. Load and pre-process DEM (La Iguana watershed, 10 m resolution)
2. Extract landslides inside the watershed boundary
3. Derive terrain covariates: slope, aspect, TWI
4. Logistic Regression with balanced sampling (equal landslide / background)
5. Predict landslide probability for all DEM cells (LR)
6. Gaussian Process conditioned on LR probabilities at landslide locations
7. Generate susceptibility maps and evaluate both models

### References
- Hosmer & Lemeshow (2000) *Applied Logistic Regression*
- Rasmussen & Williams (2006) *Gaussian Processes for Machine Learning*
- Reichenbach et al. (2018) *A review of statistically-based landslide susceptibility models*, Earth-Science Reviews
- Beven & Kirkby (1979) *A physically based, variable contributing area model*, Hydrol. Sci. Bull.

## 1. Imports and Configuration

In [ ]:
import numpy as np
# Compatibility patch: np.in1d was removed in NumPy 2.0
if not hasattr(np, 'in1d'):
    np.in1d = np.isin

import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.transform import rowcol
from rasterio.features import geometry_mask
from shapely.geometry import Point
from pysheds.grid import Grid

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score,
    f1_score, brier_score_loss, roc_curve, precision_recall_curve,
    average_precision_score
)
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel

from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

# Paths
BASE_DIR = Path('..').resolve()
DATA_DIR = BASE_DIR / 'DATA'
FIG_DIR  = BASE_DIR / 'FIGURAS'
FIG_DIR.mkdir(exist_ok=True)

np.random.seed(42)
print('Working directory:', BASE_DIR)
print('Figures directory:', FIG_DIR)

## 2. Load DEM and Watershed Boundary

The Digital Elevation Model (DEM) has a 10 m spatial resolution in MAGNA-SIRGAS / Origen Nacional (EPSG-like Colombian projection). The watershed polygon for Cuenca La Iguana is reprojected to match the DEM CRS.

In [ ]:
DEM_PATH = str(DATA_DIR / 'DEM_10m.tif')
WS_PATH  = str(DATA_DIR / 'Cuenca_Iguana.shp')
INV_PATH = str(DATA_DIR / 'MenM_VdeA.gpkg')

# Load watershed
ws = gpd.read_file(WS_PATH)

# Load DEM
with rasterio.open(DEM_PATH) as src:
    dem_crs       = src.crs
    dem_transform = src.transform
    dem_profile   = src.profile
    dem_nodata    = src.nodata
    dem_shape     = src.shape
    dem_res       = src.res
    dem_array     = src.read(1).astype(np.float32)

dem_array[dem_array == dem_nodata] = np.nan

# Reproject watershed and create mask
ws_proj = ws.to_crs(dem_crs)
ws_mask = geometry_mask(
    ws_proj.geometry,
    transform=dem_transform,
    invert=True,          # True = inside watershed
    out_shape=dem_shape
)
dem_ws = dem_array.copy()
dem_ws[~ws_mask] = np.nan

valid_cells = int(np.sum(ws_mask))
area_km2    = valid_cells * dem_res[0] * dem_res[1] / 1e6

print(f'DEM resolution:    {dem_res[0]:.1f} m x {dem_res[1]:.1f} m')
print(f'DEM shape:         {dem_shape} = {dem_shape[0]*dem_shape[1]:,} total cells')
print(f'Watershed cells:   {valid_cells:,}')
print(f'Watershed area:    {area_km2:.2f} km²')
print(f'Elevation range:   {np.nanmin(dem_ws):.0f} – {np.nanmax(dem_ws):.0f} m a.s.l.')

## 3. Load and Clip Landslide Inventory

The landslide inventory (Valle de Aburrá, 5689 events) is in WGS84 (EPSG:4326). We:
1. Strip the Z coordinate (stored as Point Z)
2. Reproject to DEM CRS
3. Spatially filter points within the watershed polygon

In [ ]:
inv = gpd.read_file(INV_PATH, layer='merged')
print(f'Total inventory (Valle de Aburrá): {len(inv):,} events')

# Remove Z component and reproject
inv['geometry'] = inv.geometry.apply(lambda g: Point(g.x, g.y))
inv_proj = inv.set_crs('EPSG:4326', allow_override=True).to_crs(dem_crs)

# Spatial filter: keep only points inside La Iguana watershed
inv_ws = (
    gpd.sjoin(inv_proj, ws_proj[['geometry']], how='inner', predicate='within')
    .drop_duplicates(subset='id')
)
print(f'Landslides in La Iguana watershed: {len(inv_ws)}')

# Save clipped inventory
out_path = str(DATA_DIR / 'Deslizamientos_Iguana.gpkg')
inv_ws[['id', 'Name', 'geometry']].to_file(out_path, driver='GPKG')
print(f'Clipped inventory saved: {out_path}')

## 4. Derive Terrain Covariates

Three topographic attributes are derived from the DEM:

| Covariate | Formula | Physical meaning |
|-----------|---------|------------------|
| **Slope** (β) | arctan(√(∂z/∂x² + ∂z/∂y²)) | Driving force for mass movements |
| **Aspect** (α) | arctan2(-∂z/∂x, ∂z/∂y) | Solar radiation, soil moisture patterns |
| **TWI** | ln(a / tan β) | Topographic proxy for soil water accumulation |

where *a* is the specific upslope contributing area computed with the D8 flow routing algorithm (O'Callaghan & Mark 1984) implemented in pysheds.

In [ ]:
def compute_slope_aspect(dem, res_x, res_y):
    """Slope (degrees) and aspect (degrees, 0=N clockwise) via central differences."""
    d = dem.copy()
    d[np.isnan(d)] = np.nanmean(d)   # fill NaN for gradient
    dy, dx = np.gradient(d, res_y, res_x)
    slope  = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2))).astype(np.float32)
    aspect = (np.degrees(np.arctan2(-dx, dy)) % 360).astype(np.float32)
    return slope, aspect

# ---- Slope and Aspect ----
print('Computing slope and aspect...')
slope, aspect = compute_slope_aspect(dem_ws, dem_res[0], dem_res[1])
slope[~ws_mask]  = np.nan
aspect[~ws_mask] = np.nan
print(f'  Slope:  {np.nanmin(slope):.1f}° – {np.nanmax(slope):.1f}°')
print(f'  Aspect: {np.nanmin(aspect):.1f}° – {np.nanmax(aspect):.1f}°')

# ---- TWI via pysheds ----
print('Computing TWI (D8 flow accumulation)...')
grid       = Grid.from_raster(DEM_PATH)
dem_g      = grid.read_raster(DEM_PATH)
pit_filled = grid.fill_pits(dem_g)
flooded    = grid.fill_depressions(pit_filled)
inflated   = grid.resolve_flats(flooded)
fdir       = grid.flowdir(inflated)
acc        = grid.accumulation(fdir).astype(np.float64)

# Slope in radians for TWI formula
rx = abs(grid.affine.a); ry = abs(grid.affine.e)
da = np.array(dem_g).astype(np.float64)
da[da == dem_g.nodata] = np.nan
df_dem = da.copy(); df_dem[np.isnan(df_dem)] = np.nanmean(df_dem)
dy_g, dx_g = np.gradient(df_dem, ry, rx)
sr = np.arctan(np.sqrt(dx_g**2 + dy_g**2))
sr[sr < 0.001] = 0.001          # avoid division by zero on flat terrain

# Specific contributing area: a = (acc_cells + 1) * cell_area / contour_length
a_sp = (np.array(acc) + 1) * rx * ry / rx
twi  = np.log(a_sp / np.tan(sr)).astype(np.float32)

# Resize if pysheds grid differs from DEM grid
if twi.shape != dem_shape:
    from scipy.ndimage import zoom
    twi = zoom(twi, (dem_shape[0]/twi.shape[0], dem_shape[1]/twi.shape[1]), order=1)

twi[~ws_mask] = np.nan
print(f'  TWI: {np.nanmin(twi):.2f} – {np.nanmax(twi):.2f}')

## 5. Visualize Terrain Covariates

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Terrain Covariates – Cuenca La Iguana', fontsize=14, fontweight='bold')

im0 = axes[0,0].imshow(dem_ws,  cmap='terrain',  interpolation='nearest')
axes[0,0].set_title('Elevation (m a.s.l.)', fontweight='bold')
plt.colorbar(im0, ax=axes[0,0], label='m')

im1 = axes[0,1].imshow(slope, cmap='YlOrRd', vmin=0, vmax=60, interpolation='nearest')
axes[0,1].set_title('Slope (degrees)', fontweight='bold')
plt.colorbar(im1, ax=axes[0,1], label='°')

im2 = axes[1,0].imshow(aspect, cmap='hsv', vmin=0, vmax=360, interpolation='nearest')
axes[1,0].set_title('Aspect (degrees from North, clockwise)', fontweight='bold')
plt.colorbar(im2, ax=axes[1,0], label='°')

twi_clip = np.clip(twi, np.nanpercentile(twi,2), np.nanpercentile(twi,98))
im3 = axes[1,1].imshow(twi_clip, cmap='Blues', interpolation='nearest')
axes[1,1].set_title('Topographic Wetness Index (TWI)', fontweight='bold')
plt.colorbar(im3, ax=axes[1,1], label='TWI')

for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig(str(FIG_DIR / 'Fig1_terrain_covariates.png'), dpi=200, bbox_inches='tight')
plt.show()

## 6. Sampling Strategy

We use a **balanced sampling** strategy:
- **Presence** (label = 1): the 583 landslide cells
- **Absence** (label = 0): an equal number of randomly selected non-landslide cells within the watershed

Random selection ensures absence cells are not collocated with known landslide cells.

In [ ]:
def extract_at_points(geoms, arrays, transform, shape):
    """Extract raster values at point geometries."""
    rows = []
    data = {k: [] for k in arrays}
    for geom in geoms:
        r, c = rowcol(transform, geom.x, geom.y)
        if 0 <= r < shape[0] and 0 <= c < shape[1]:
            vals = {k: arr[r, c] for k, arr in arrays.items()}
            if not any(np.isnan(v) for v in vals.values()):
                for k, v in vals.items(): data[k].append(v)
                rows.append((r, c))
    return pd.DataFrame(data), rows

# Extract covariate values at landslide locations
arrs = {'slope': slope, 'aspect': aspect, 'twi': twi}
ls_df, ls_cells = extract_at_points(inv_ws.geometry, arrs, dem_transform, dem_shape)
ls_df['label'] = 1
ls_cells_set = set(map(tuple, ls_cells))
n_ls = len(ls_df)
print(f'Landslide samples with valid covariates: {n_ls}')

# Generate background (absence) sample
valid_r, valid_c = np.where(ws_mask & ~np.isnan(slope) & ~np.isnan(twi))
idx = np.random.choice(len(valid_r), size=min(len(valid_r), n_ls*5), replace=False)
bg = []
for i in idx:
    r, c = int(valid_r[i]), int(valid_c[i])
    if (r, c) not in ls_cells_set:
        bg.append({'slope': slope[r,c], 'aspect': aspect[r,c], 'twi': twi[r,c], 'label': 0})
    if len(bg) >= n_ls: break
bg_df = pd.DataFrame(bg[:n_ls])

dataset = pd.concat([ls_df[['slope','aspect','twi','label']], bg_df], ignore_index=True)
print(f'Total dataset: {len(dataset)} samples | {dataset.label.value_counts().to_dict()}')

# Covariate distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Covariate Distributions: Landslide vs Background', fontweight='bold')
feat_names = ['Slope (°)', 'Aspect (°)', 'TWI']
for i, (ax, fname) in enumerate(zip(axes, feat_names)):
    feats = ['slope','aspect','twi']
    ls_v = dataset[dataset.label==1][feats[i]].values
    bg_v = dataset[dataset.label==0][feats[i]].values
    ax.hist(bg_v, bins=40, alpha=0.6, color='steelblue', label='Background', density=True)
    ax.hist(ls_v, bins=40, alpha=0.6, color='firebrick', label='Landslide',  density=True)
    ax.set_xlabel(fname); ax.set_ylabel('Density')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(FIG_DIR / 'Fig7_covariate_distributions.png'), dpi=200, bbox_inches='tight')
plt.show()

## 7. Logistic Regression Model

Logistic Regression models the probability of landslide occurrence as:

$$P(Y=1|\mathbf{x}) = \frac{1}{1 + e^{-(\beta_0 + \boldsymbol{\beta}^\top \mathbf{x})}}$$

with L2 regularization ($C=1$). Features are standardized before fitting. Model selection uses 5-fold stratified cross-validation.

In [ ]:
feats = ['slope', 'aspect', 'twi']
X = dataset[feats].values
y = dataset['label'].values

# Stratified 70/30 split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Standardize
scaler   = StandardScaler()
X_tr_s   = scaler.fit_transform(X_tr)
X_te_s   = scaler.transform(X_te)

# Fit Logistic Regression
lr = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs', max_iter=1000, random_state=42)
lr.fit(X_tr_s, y_tr)

# Predictions
y_prob_te = lr.predict_proba(X_te_s)[:, 1]
y_pred_te = (y_prob_te >= 0.5).astype(int)

# Cross-validation
cv_auc = cross_val_score(lr, scaler.transform(X), y,
                          cv=StratifiedKFold(5, shuffle=True, random_state=42),
                          scoring='roc_auc')

print('=== Logistic Regression – Training Set Performance ===')
print(f'  AUC (test set):       {roc_auc_score(y_te, y_prob_te):.4f}')
print(f'  AUC (5-fold CV):      {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')
print(f'  Accuracy:             {accuracy_score(y_te, y_pred_te):.4f}')
print(f'  F1:                   {f1_score(y_te, y_pred_te):.4f}')
print(f'  Brier score:          {brier_score_loss(y_te, y_prob_te):.4f}')
print(f'\nModel coefficients (standardized):')
for feat, coef in zip(feats, lr.coef_[0]):
    print(f'  {feat:8s}: {coef:+.4f}')
print(f'  Intercept: {lr.intercept_[0]:+.4f}')

## 8. Predict Landslide Probability – Logistic Regression (All Cells)

In [ ]:
# Valid cells mask: inside watershed + all covariates available
valid_mask = ws_mask & ~np.isnan(slope) & ~np.isnan(aspect) & ~np.isnan(twi)
r_all, c_all = np.where(valid_mask)

# Build feature matrix for all valid cells
X_all   = np.column_stack([slope[r_all, c_all], aspect[r_all, c_all], twi[r_all, c_all]])
X_all_s = scaler.transform(X_all)

print(f'Predicting for {len(X_all):,} cells...')
prob_lr_all = lr.predict_proba(X_all_s)[:, 1]

# Reconstruct raster
prob_map_lr = np.full(dem_shape, np.nan, np.float32)
prob_map_lr[r_all, c_all] = prob_lr_all

print(f'LR probability range: {np.nanmin(prob_map_lr):.4f} – {np.nanmax(prob_map_lr):.4f}')
print(f'LR probability mean:  {np.nanmean(prob_map_lr):.4f}')

## 9. Gaussian Process Model

### Rationale
The GP is conditioned on the LR-predicted probabilities at known landslide locations. This two-stage approach:
1. Uses LR to translate covariates into a probabilistic signal
2. Uses GP to model the non-linear, spatially correlated structure of that signal across covariate space

### Kernel
We use a **Matérn** (ν=1.5) covariance kernel which produces once-differentiable sample functions — appropriate for spatially smooth but not overly smooth susceptibility fields:

$$k(\mathbf{x}, \mathbf{x}') = \sigma_f^2 \left(1 + \frac{\sqrt{3}\,\|\mathbf{x}-\mathbf{x}'\|}{l}\right) \exp\left(-\frac{\sqrt{3}\,\|\mathbf{x}-\mathbf{x}'\|}{l}\right) + \sigma_n^2\,\delta(\mathbf{x},\mathbf{x}')$$

Parameters $(\sigma_f, l, \sigma_n)$ are optimised by maximising the log marginal likelihood.

In [ ]:
# Extract LR-predicted probabilities at landslide cell locations
ls_r_list, ls_c_list = [], []
for geom in inv_ws.geometry:
    r, c = rowcol(dem_transform, geom.x, geom.y)
    ls_r_list.append(r); ls_c_list.append(c)

gp_r, gp_c, gp_y = [], [], []
seen = set()
for r, c in zip(ls_r_list, ls_c_list):
    if 0 <= r < dem_shape[0] and 0 <= c < dem_shape[1] and valid_mask[r,c] and (r,c) not in seen:
        seen.add((r, c))
        gp_r.append(r); gp_c.append(c)
        gp_y.append(float(prob_map_lr[r, c]))

gp_r = np.array(gp_r); gp_c = np.array(gp_c); gp_y = np.array(gp_y)

# Feature matrix for GP (same covariates as LR)
X_gp = np.column_stack([slope[gp_r, gp_c], aspect[gp_r, gp_c], twi[gp_r, gp_c]])
ok = ~np.any(np.isnan(X_gp), axis=1)
X_gp = X_gp[ok]; gp_y = gp_y[ok]
print(f'GP training points: {len(X_gp)}')
print(f'Target range (LR prob at LS cells): {gp_y.min():.4f} – {gp_y.max():.4f}')

# Standardize GP features
scaler_gp = StandardScaler()
X_gp_s    = scaler_gp.fit_transform(X_gp)

# Subsample if n > MAX_GP (GP scales as O(n³))
MAX_GP = 400
if len(X_gp_s) > MAX_GP:
    idx_sub = np.random.choice(len(X_gp_s), MAX_GP, replace=False)
    X_fit = X_gp_s[idx_sub]; y_fit = gp_y[idx_sub]
    print(f'Subsampled to {MAX_GP} points (GP is O(n³))')
else:
    X_fit = X_gp_s; y_fit = gp_y

# Kernel: ConstantKernel × Matérn(ν=1.5) + WhiteKernel (nugget)
kernel = (ConstantKernel(1.0, (0.01, 10.0))
          * Matern(length_scale=1.0, length_scale_bounds=(0.01, 10.0), nu=1.5)
          + WhiteKernel(0.01, (1e-4, 0.5)))

print('Fitting Gaussian Process (hyperparameter optimisation)...')
gp = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=5,
    normalize_y=True,
    random_state=42
)
gp.fit(X_fit, y_fit)
print('Optimised kernel:', gp.kernel_)

## 10. Predict with GP (All Watershed Cells)

GP prediction returns both the **posterior mean** $\mu(\mathbf{x}^*)$ and **posterior standard deviation** $\sigma(\mathbf{x}^*)$ at each cell. Predictions are computed in mini-batches for memory efficiency.

In [ ]:
X_all_gp_s = scaler_gp.transform(X_all)
n = len(X_all_gp_s)
gp_mu = np.zeros(n, np.float32)
gp_sg = np.zeros(n, np.float32)

BATCH = 5000
print(f'Predicting GP for {n:,} cells (batch={BATCH})...')
for i in range(0, n, BATCH):
    if i % (50*BATCH) == 0:
        print(f'  {i:>8,} / {n:,}  ({100*i/n:.0f}%)')
    mu, sg = gp.predict(X_all_gp_s[i:i+BATCH], return_std=True)
    gp_mu[i:i+BATCH] = mu.astype(np.float32)
    gp_sg[i:i+BATCH] = sg.astype(np.float32)

gp_mu = np.clip(gp_mu, 0, 1)   # clip to valid probability range

# Reconstruct rasters
prob_map_gp_mu = np.full(dem_shape, np.nan, np.float32)
prob_map_gp_sg = np.full(dem_shape, np.nan, np.float32)
prob_map_gp_mu[r_all, c_all] = gp_mu
prob_map_gp_sg[r_all, c_all] = gp_sg

print(f'GP mean: {np.nanmin(prob_map_gp_mu):.4f} – {np.nanmax(prob_map_gp_mu):.4f}')
print(f'GP std:  {np.nanmin(prob_map_gp_sg):.4f} – {np.nanmax(prob_map_gp_sg):.4f}')

## 11. Model Evaluation

Performance is evaluated on the **full watershed grid** against the binary landslide occurrence map (1 = known landslide cell, 0 = all other valid cells). Note the strong class imbalance (583 landslide cells out of 521,586 valid cells ≈ 0.11%), which explains low precision and F1 at the 0.5 threshold — AUC and AP are the primary discrimination metrics.

In [ ]:
# Ground-truth map
label_map = np.zeros(dem_shape, np.int8)
for r, c in zip(ls_r_list, ls_c_list):
    if 0 <= r < dem_shape[0] and 0 <= c < dem_shape[1]:
        label_map[r, c] = 1
y_true = label_map[r_all, c_all]

def compute_metrics(y_true, y_prob, name, thr=0.5):
    yp = (y_prob >= thr).astype(int)
    return {
        'Model' : name,
        'AUC'   : round(roc_auc_score(y_true, y_prob), 4),
        'AP'    : round(average_precision_score(y_true, y_prob), 4),
        'Acc'   : round(accuracy_score(y_true, yp), 4),
        'Prec'  : round(precision_score(y_true, yp, zero_division=0), 4),
        'Rec'   : round(recall_score(y_true, yp, zero_division=0), 4),
        'F1'    : round(f1_score(y_true, yp, zero_division=0), 4),
        'Brier' : round(brier_score_loss(y_true, y_prob), 4),
    }

m_lr = compute_metrics(y_true, prob_lr_all, 'Logistic Regression')
m_gp = compute_metrics(y_true, gp_mu,       'Gaussian Process')
df_met = pd.DataFrame([m_lr, m_gp]).set_index('Model')

print('=== Performance Metrics – Full Watershed Grid ===')
print(df_met.to_string())
df_met.to_csv(str(DATA_DIR / 'metrics.csv'))

## 12. ROC and PR Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Evaluation – La Iguana Watershed', fontsize=13, fontweight='bold')

ax = axes[0]
for lbl, yp, col in [('LR', prob_lr_all, 'steelblue'), ('GP', gp_mu, 'darkorange')]:
    fpr, tpr, _ = roc_curve(y_true, yp)
    ax.plot(fpr, tpr, color=col, lw=2, label=f'{lbl} (AUC={roc_auc_score(y_true,yp):.3f})')
ax.plot([0,1],[0,1],'k--',lw=1)
ax.set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC Curves')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
for lbl, yp, col in [('LR', prob_lr_all, 'steelblue'), ('GP', gp_mu, 'darkorange')]:
    pr, rc, _ = precision_recall_curve(y_true, yp)
    ax.plot(rc, pr, color=col, lw=2, label=f'{lbl} (AP={average_precision_score(y_true,yp):.3f})')
ax.set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curves')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
metric_cols = ['AUC','AP','Acc','F1']
x_pos = np.arange(len(metric_cols)); w = 0.35
ax.bar(x_pos-w/2, [m_lr[m] for m in metric_cols], w, label='LR', color='steelblue', ec='black')
ax.bar(x_pos+w/2, [m_gp[m] for m in metric_cols], w, label='GP', color='darkorange', ec='black')
ax.set_xticks(x_pos); ax.set_xticklabels(metric_cols)
ax.set(ylim=(0,1.15), ylabel='Score', title='Performance Comparison')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
for bar in ax.patches:
    ax.annotate(f'{bar.get_height():.3f}',
                (bar.get_x()+bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(str(FIG_DIR / 'Fig2_model_evaluation.png'), dpi=200, bbox_inches='tight')
plt.show()

## 13. Susceptibility Maps

In [ ]:
susc_cmap = LinearSegmentedColormap.from_list(
    'susc', ['#1a9641','#a6d96a','#ffffbf','#fdae61','#d7191c'], N=256
)

# Landslide pixel positions for overlay
px_x, px_y = [], []
for geom in inv_ws.geometry:
    r, c = rowcol(dem_transform, geom.x, geom.y)
    px_x.append(int(np.clip(c, 0, dem_shape[1]-1)))
    px_y.append(int(np.clip(r, 0, dem_shape[0]-1)))
px_x, px_y = np.array(px_x), np.array(px_y)

# ---- Fig 3: LR susceptibility ----
fig, ax = plt.subplots(figsize=(9, 10))
im = ax.imshow(prob_map_lr, cmap=susc_cmap, vmin=0, vmax=1, interpolation='nearest')
ax.scatter(px_x, px_y, s=4, c='black', marker='.', alpha=0.5, label=f'Landslides (n={len(inv_ws)})')
plt.colorbar(im, ax=ax, label='P(landslide)', fraction=0.035, pad=0.04)
ax.set_title('Logistic Regression – Landslide Susceptibility\nCuenca La Iguana', fontsize=13, fontweight='bold')
ax.legend(loc='lower right'); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig(str(FIG_DIR / 'Fig3_LR_susceptibility.png'), dpi=200, bbox_inches='tight')
plt.show()

# ---- Fig 4: GP mean + std ----
fig, axes = plt.subplots(1, 2, figsize=(17, 9))
fig.suptitle('Gaussian Process – Landslide Susceptibility\nCuenca La Iguana', fontsize=13, fontweight='bold')
im0 = axes[0].imshow(prob_map_gp_mu, cmap=susc_cmap, vmin=0, vmax=1, interpolation='nearest')
axes[0].scatter(px_x, px_y, s=4, c='black', marker='.', alpha=0.5, label=f'Landslides (n={len(inv_ws)})')
plt.colorbar(im0, ax=axes[0], label='Mean P(landslide)', fraction=0.035, pad=0.04)
axes[0].set_title('GP Mean Predicted Probability', fontweight='bold')
axes[0].legend(loc='lower right'); axes[0].set_xticks([]); axes[0].set_yticks([])

std_c = np.clip(prob_map_gp_sg, 0, np.nanpercentile(prob_map_gp_sg, 98))
im1 = axes[1].imshow(std_c, cmap='plasma', interpolation='nearest')
axes[1].scatter(px_x, px_y, s=4, c='white', marker='.', alpha=0.5)
plt.colorbar(im1, ax=axes[1], label='Std. Deviation', fraction=0.035, pad=0.04)
axes[1].set_title('GP Prediction Uncertainty (Std. Dev.)', fontweight='bold')
axes[1].set_xticks([]); axes[1].set_yticks([])
plt.tight_layout()
plt.savefig(str(FIG_DIR / 'Fig4_GP_susceptibility_mean_std.png'), dpi=200, bbox_inches='tight')
plt.show()

# ---- Fig 5: Side-by-side comparison ----
fig, axes = plt.subplots(1, 2, figsize=(17, 9))
fig.suptitle('Susceptibility Comparison – Cuenca La Iguana', fontsize=13, fontweight='bold')
for ax, pmap, title in [
    (axes[0], prob_map_lr,    'Logistic Regression'),
    (axes[1], prob_map_gp_mu, 'Gaussian Process (Mean)'),
]:
    im = ax.imshow(pmap, cmap=susc_cmap, vmin=0, vmax=1, interpolation='nearest')
    ax.scatter(px_x, px_y, s=4, c='black', marker='.', alpha=0.5, label=f'n={len(inv_ws)}')
    plt.colorbar(im, ax=ax, label='P(landslide)', fraction=0.035, pad=0.04)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig(str(FIG_DIR / 'Fig5_comparison_LR_GP.png'), dpi=200, bbox_inches='tight')
plt.show()

## 14. Susceptibility Class Distribution

In [ ]:
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
class_labels = ['Very Low', 'Low', 'Moderate', 'High', 'Very High']
cls_colors   = ['#1a9641', '#a6d96a', '#ffffbf', '#fdae61', '#d7191c']

def get_class_pcts(pmap):
    valid = pmap[~np.isnan(pmap)]
    counts = [np.sum((valid >= lo) & (valid < hi)) for lo, hi in zip(bins[:-1], bins[1:])]
    counts[-1] += np.sum(valid == 1.0)
    total = sum(counts)
    return [100*c/total for c in counts]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Susceptibility Class Distribution – Cuenca La Iguana', fontsize=13, fontweight='bold')
for ax, pmap, name in [
    (axes[0], prob_map_lr,    'Logistic Regression'),
    (axes[1], prob_map_gp_mu, 'Gaussian Process'),
]:
    pcts = get_class_pcts(pmap)
    bars = ax.bar(class_labels, pcts, color=cls_colors, edgecolor='black', linewidth=0.5)
    for bar, pct in zip(bars, pcts):
        ax.annotate(f'{pct:.1f}%', (bar.get_x()+bar.get_width()/2, bar.get_height()),
                    ha='center', va='bottom', fontsize=9)
    ax.set(ylabel='Watershed Area (%)', title=name, ylim=(0, max(pcts)*1.18))
    ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(str(FIG_DIR / 'Fig6_susceptibility_classes.png'), dpi=200, bbox_inches='tight')
plt.show()

## 15. Save Output Rasters

In [ ]:
def save_raster(arr, path, profile):
    p = profile.copy()
    p.update(dtype='float32', count=1, nodata=-9999.0, compress='lzw')
    a = arr.copy().astype(np.float32); a[np.isnan(a)] = -9999.0
    with rasterio.open(path, 'w', **p) as dst: dst.write(a, 1)
    print(f'Saved: {path}')

save_raster(slope,          str(DATA_DIR / 'slope.tif'),                dem_profile)
save_raster(aspect,         str(DATA_DIR / 'aspect.tif'),               dem_profile)
save_raster(twi,            str(DATA_DIR / 'twi.tif'),                  dem_profile)
save_raster(prob_map_lr,    str(DATA_DIR / 'susceptibility_LR.tif'),    dem_profile)
save_raster(prob_map_gp_mu, str(DATA_DIR / 'susceptibility_GP_mean.tif'), dem_profile)
save_raster(prob_map_gp_sg, str(DATA_DIR / 'susceptibility_GP_std.tif'),  dem_profile)

print('\nFinal metrics summary:')
print(df_met.to_string())